# Expand S09 to top 400 on one Colab Pro GPU

Keep the best 0.1626 submission. This run changes only rerank pool size and fixes output to 120 documents / 120 chunks / 6 per document / IoU 0.8. It reuses verified exact old scores, scores new pairs on CUDA, validates against the full structural store, then packages a submission. No score improvement is guaranteed.

Required: old S09 bundle and completed worker outputs; full BM25 and BGE candidate stores; full structural chunk store; official query and links Parquet. The old bundle alone cannot supply new evidence. Use a fresh runtime. Preparation scans source data on CPU; allow local disk for the source store, packets and weights. All cells run manually. No website upload is automated.

Input preflight runs before large copies. A semantic store with the same batch count is not the structural S09 source. Missing source data stops this notebook with diagnostics; no metadata is rewritten. Copy progress is printed every 50 files, with a 30-second activity message.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, zipfile, tempfile, shutil, sys, subprocess, os
ROOT = Path('/content/drive/MyDrive/ViBioMIR')
BUNDLE = ROOT / 'stage9_rerank_colab.zip'
WORKERS = ROOT / 'stage9_rerank_workers'
DATA_ZIP = ROOT / 'Data.zip'
QUERY_FILE = ROOT / 'Data/official/ViBioMIR/query.parquet'
LINKS_FILE = ROOT / 'Data/official/ViBioMIR/links_corpus_vi_en_zh.parquet'
LINKS_ID_COLUMN = 'id'  # Change only if the official file uses another ID column.
ARTIFACTS_ZIP = ROOT / 'artifacts.zip'  # optional source of the S09 bundle/worker files
RUN_MISSING_RERANK = False  # True requires CUDA; missing scores may take substantial time
RERANK_BACKUP = ROOT / 'stage9_submission_resume'
EXPECTED_LINKS_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
LIMITS = dict(max_docs=120, max_chunks=120, max_chunks_per_doc=6, overlap_iou_threshold=0.8)
EXPECTED_QUERIES = 1200
OUTPUT_PARENT = ROOT / 'submission_trials'
print('Root files:')
assert ROOT.is_dir(), f'Drive directory missing: {ROOT}'
for p in ROOT.iterdir():
    print(p.name)
print('Bundle:', BUNDLE, 'exists:', BUNDLE.is_file())
print('Worker folder:', WORKERS, 'exists:', WORKERS.is_dir())

RERANK_TOP_K = 400  # choose 300 BEFORE starting a new run if time is tight
STOP_AFTER_QUERIES = 5  # timing pilot, then None; rerun the GPU cell
ARTIFACTS_DIR = ROOT / 'artifacts'
CHUNK_STORE_OVERRIDE = None  # Set to Path('/content/drive/MyDrive/ViBioMIR/semantic_parallel_run01/merged') only if its snapshot matches S09.
RUN_DRIVE = ROOT / f'stage9_expand_{RERANK_TOP_K}_120_120_v1'
LOCAL_INPUTS = Path('/content/stage9_expand_inputs')
NEW_PREPARED = Path('/content/stage9_expand_prepared')
NEW_WORKERS = Path('/content/stage9_expand_workers')
assert RERANK_TOP_K in (300, 400)
RUN_DRIVE.mkdir(parents=True, exist_ok=True)


In [ ]:
# Extract only a nested S09 bundle if present, not the full corpus archive.
if not BUNDLE.is_file() and ARTIFACTS_ZIP.is_file():
    with zipfile.ZipFile(ARTIFACTS_ZIP) as archive:
        matches = [n for n in archive.namelist() if Path(n).name == 'stage9_rerank_colab.zip']
        assert len(matches) <= 1, 'Multiple S09 bundles in artifacts.zip; select the matching bundle explicitly.'
        if matches:
            local_bundle = Path(tempfile.mkdtemp(prefix='s09_bundle_', dir='/content')) / 'stage9_rerank_colab.zip'
            with archive.open(matches[0]) as src, local_bundle.open('wb') as dst:
                shutil.copyfileobj(src, dst)
            BUNDLE = local_bundle
assert BUNDLE.is_file(), f'Missing S09 bundle: {BUNDLE}. Upload the matching stage9_rerank_colab.zip from the completed S09 run; a Stage-5 ZIP is not suitable.'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyarrow==22.0.0'], check=True)
PROJECT = Path(tempfile.mkdtemp(prefix='vibiomir_trial_', dir='/content'))
def digest(data):
    return hashlib.sha256(data).hexdigest()
def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
with zipfile.ZipFile(BUNDLE) as z:
    meta = json.loads(z.read('BUNDLE.json'))
    assert digest(canonical({k:v for k,v in meta.items() if k != 'signature'}).encode()) == meta['signature']
    assert len(z.namelist()) == len(set(z.namelist()))
    assert set(z.namelist()) == set(meta['files']) | {'BUNDLE.json'}
    assert shutil.disk_usage('/content').free > sum(i.file_size for i in z.infolist()) + 1024**3
    for name, expected in meta['files'].items():
        target = (PROJECT / name).resolve()
        assert PROJECT.resolve() in target.parents, 'Unsafe bundle path'
        payload = z.read(name)
        assert digest(payload) == expected, f'Bundle checksum mismatch: {name}'
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(payload)
# Use exactly the frozen source that created the scores.
assert not any(n == 'vibiomir' or n.startswith('vibiomir.') for n in sys.modules), 'Restart runtime to avoid mixing source versions.'
sys.path.insert(0, str(PROJECT))
from vibiomir.stage9_pipeline import prepared, load_packet, verify_scores, _worker_identity, import_worker
from vibiomir.stage9_core import select
from vibiomir.retrieval_contracts import checked, queries
from vibiomir.io import file_hash, text_hash, canonical_json
state = prepared(PROJECT)
assert state['identity']['scope'] == 'all_queries', 'A pilot bundle cannot produce a full submission.'
assert len(state['packets']) == EXPECTED_QUERIES
assert all(type(LIMITS[k]) is int and LIMITS[k] > 0 for k in ('max_docs','max_chunks','max_chunks_per_doc'))
assert 0 < LIMITS['overlap_iou_threshold'] <= 1
print('Verified prepared queries:', len(state['packets']))


In [ ]:
# Run before copying corpus files or loading GPU models.
from vibiomir.retrieval_contracts import chunk_snapshot
expected_source = state['identity']['source']
print('Required S09 source:', json.dumps(expected_source, indent=2), flush=True)
locations = [
    LOCAL_INPUTS / 'chunks_stage5_full_structure',
    ARTIFACTS_DIR / 'chunks_stage5_full_structure',
    ROOT / 'chunks_stage5_full_structure',
    ROOT / 'stage5_full_structure',
    ROOT / 'semantic_parallel_run01/merged',
]
if CHUNK_STORE_OVERRIDE is not None:
    locations = [Path(CHUNK_STORE_OVERRIDE)]
matched = None
for folder in locations:
    if not folder.is_dir():
        print('NOT FOUND:', folder, flush=True)
        continue
    try:
        snapshot = chunk_snapshot(folder)
        differences = {k: {'actual': snapshot.get(k), 'expected': v}
                       for k, v in expected_source.items() if snapshot.get(k) != v}
        if differences:
            print('DIFFERENT SOURCE:', folder, json.dumps(differences), flush=True)
        else:
            matched = folder
            print('MATCH:', folder, flush=True)
            break
    except (ValueError, KeyError, OSError) as exc:
        print('INCOMPLETE OR INVALID:', folder, str(exc), flush=True)
if matched is not None:
    CHUNK_STORE_OVERRIDE = matched
else:
    # Check only archive metadata, without copying/extracting the corpus.
    archive_match = False
    if CHUNK_STORE_OVERRIDE is None and ARTIFACTS_ZIP.is_file():
        with zipfile.ZipFile(ARTIFACTS_ZIP) as z:
            names = [n for n in z.namelist()
                     if n.endswith('chunks_stage5_full_structure/manifest.json')]
            if len(names) == 1:
                prefix = names[0][:-len('manifest.json')]
                if prefix + 'VALIDATION.json' in z.namelist():
                    archive_match = (
                        digest(z.read(names[0])) == expected_source['manifest_sha256']
                        and digest(z.read(prefix + 'VALIDATION.json'))
                        == expected_source['validation_sha256'])
    if not archive_match:
        raise ValueError(
            'No matching complete structural source found. Nothing has been copied. '
            'Restore the original 5,512,624-chunk source and set CHUNK_STORE_OVERRIDE. '
            'The 9,036,279-chunk semantic source and S09 worker folders cannot replace it. '
            'Keep the existing best submission; this notebook cannot reconstruct missing source text.')
    print('Matching source metadata found in artifacts.zip.', flush=True)


In [ ]:
import pyarrow.parquet as pq
def official_file(path, basenames):
    if path.is_file():
        return path
    assert DATA_ZIP.is_file(), f'Missing {path}. Set the path or provide Data.zip.'
    with zipfile.ZipFile(DATA_ZIP) as z:
        matches = [n for n in z.namelist() if Path(n).name in basenames and 'official' in Path(n).parts]
        assert len(matches) == 1, f'Expected one official file, found {matches}. Extract the intended file and set its path.'
        target = PROJECT / 'official' / Path(matches[0]).name
        target.parent.mkdir(exist_ok=True)
        with z.open(matches[0]) as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)
        return target
query_path = official_file(QUERY_FILE, {'query.parquet'})
links_path = official_file(LINKS_FILE, {'links_corpus_vi_en_zh.parquet', 'links_corpus.parquet'})
official_queries = queries(query_path)
assert file_hash(query_path) == state['identity']['query_file_sha256'], 'Query snapshot differs from frozen run.'
assert [q['id'] for q in official_queries] == [p['id'] for p in state['packets']]
assert file_hash(links_path) == EXPECTED_LINKS_SHA256, 'Official links differ from the verified baseline snapshot; verify the file before continuing.'
pf = pq.ParquetFile(links_path)
assert LINKS_ID_COLUMN in pf.schema_arrow.names, f'Set LINKS_ID_COLUMN. Available: {pf.schema_arrow.names}'
allowed = set()
for batch in pf.iter_batches(columns=[LINKS_ID_COLUMN]):
    for value in batch.column(0).to_pylist():
        assert type(value) is int, 'Official IDs must be integers.'
        allowed.add(value)
assert allowed


In [ ]:
# Search only expected worker locations; never unpack the full corpus.
restored_workers = PROJECT / 'restored_workers'
restored_workers.mkdir()
worker_roots, worker_manifests = {}, {}
def locate_worker(i):
    for parent in (RERANK_BACKUP, WORKERS):
        folder = parent / f'worker_{i}'
        if (folder / 'manifest.json').is_file():
            return folder
    for archive in (WORKERS / f'stage9_worker_{i}.zip', ROOT / f'stage9_worker_{i}.zip'):
        if archive.is_file():
            import_worker(PROJECT, archive, restored_workers)
            return restored_workers / f'worker_{i}'
    if ARTIFACTS_ZIP.is_file():
        with zipfile.ZipFile(ARTIFACTS_ZIP) as z:
            matches = [n for n in z.namelist() if Path(n).name == f'stage9_worker_{i}.zip']
            assert len(matches) <= 1, 'Ambiguous worker archives; extract the matching one and set WORKERS.'
            if matches:
                archive = PROJECT / f'stage9_worker_{i}.zip'
                with z.open(matches[0]) as src, archive.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
                import_worker(PROJECT, archive, restored_workers)
                return restored_workers / f'worker_{i}'
            prefixes = [n[:-len('manifest.json')] for n in z.namelist()
                        if n.endswith(f'stage9_rerank_workers/worker_{i}/manifest.json')]
            assert len(prefixes) <= 1, 'Ambiguous worker folders in artifacts.zip.'
            if prefixes:
                prefix = prefixes[0]
                folder = restored_workers / f'worker_{i}'
                folder.mkdir(exist_ok=True)
                names = ['manifest.json'] + [f"query-{p['id']}.json" for p in state['packets'] if p['worker_id'] == i]
                for name in names:
                    if prefix + name in z.namelist():
                        with z.open(prefix + name) as src, (folder / name).open('wb') as dst:
                            shutil.copyfileobj(src, dst)
                return folder
    return None

if RUN_MISSING_RERANK:
    os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    str(PROJECT / 'requirements-stage9-colab.txt')], check=True)
    import torch
    assert torch.cuda.is_available(), 'Select a GPU runtime, then rerun from a fresh runtime.'
    from vibiomir.stage9_reranker import recommended_runtime
    preset = recommended_runtime(torch.cuda.get_device_properties(0).total_memory / 1024**3)
    print('GPU:', torch.cuda.get_device_name(0), 'budgets:', preset)

for i in range(state['identity']['settings']['workers']):
    folder = locate_worker(i)
    ident = _worker_identity(state, i)
    expected_ids = [p['id'] for p in state['packets'] if p['worker_id'] == i]
    manifest = json.loads((folder / 'manifest.json').read_text()) if folder else None
    if manifest:
        assert manifest['identity'] == ident and manifest['signature'] == text_hash(canonical_json(ident)), 'Worker belongs to another run.'
    complete = manifest is not None and manifest['state'] == 'complete' and manifest.get('completed_query_ids') == expected_ids
    files_present = folder is not None and all((folder / f'query-{q}.json').is_file() for q in expected_ids)
    if not (complete and files_present):
        assert RUN_MISSING_RERANK, f'Worker {i} missing/incomplete. Supply matching completed worker ZIPs, or enable RUN_MISSING_RERANK on CUDA.'
        local = PROJECT / 'resume_workers' / f'worker_{i}'
        local.mkdir(parents=True, exist_ok=True)
        if folder:
            for entry in state['packets']:
                if entry['worker_id'] != i: continue
                path = folder / f"query-{entry['id']}.json"
                if path.is_file():
                    packet = load_packet(PROJECT, entry)
                    verify_scores(checked(path), packet, entry, state, text_hash(canonical_json(ident)))
                    shutil.copy2(path, local / path.name)
        command = [sys.executable, '-u', '-m', 'vibiomir.stage9_pipeline', 'rerank-worker',
                   '--prepared-dir', str(PROJECT), '--output-dir', str(local), '--worker-id', str(i),
                   '--backup-dir', str(RERANK_BACKUP / f'worker_{i}'),
                   '--batch-size', str(preset['batch_size']), '--padded-tokens', str(preset['padded_tokens'])]
        subprocess.run(command, cwd=PROJECT, check=True)
        folder = local
        manifest = json.loads((folder / 'manifest.json').read_text())
    assert manifest['identity'] == ident and manifest['signature'] == text_hash(canonical_json(ident))
    assert manifest['state'] == 'complete' and manifest['completed_query_ids'] == expected_ids
    worker_roots[i], worker_manifests[i] = folder, manifest
print('Workers ready; every query score will be verified before export.')


In [ ]:
# Load this notebook's support code; frozen producer files are unchanged.
HELPER_SOURCE = "\"\"\"Verified score reuse and submission packaging for an expanded S09 pool.\"\"\"\nimport json\nfrom pathlib import Path\nimport sqlite3\nimport zipfile\n\nfrom vibiomir.io import canonical_json, file_hash, text_hash, write_json, writer_lock\nfrom vibiomir.retrieval_contracts import checked, open_run, require\nfrom vibiomir.stage9_core import score_key\nfrom vibiomir.stage9_pipeline import prepared, load_packet, verify_scores, _worker_identity\n\n\ndef seed_cache(old_dir, old_workers, new_dir, output, worker_id=0):\n    \"\"\"Import verified exact query/passage/policy scores; never reuse chunk-ID alone.\"\"\"\n    old, new = prepared(old_dir), prepared(new_dir)\n    policy = new['identity']['settings']['reranker']\n    require(old['identity']['settings']['reranker'] == policy, 'Reranker policy differs')\n    require(old['identity']['source'] == new['identity']['source'], 'Corpus snapshot differs')\n    require(old['identity']['code'] == new['identity']['code'], 'Frozen scoring code differs')\n    require(old['identity']['query_file_sha256'] == new['identity']['query_file_sha256'], 'Query snapshot differs')\n    old_entries = {p['id']: p for p in old['packets']}\n    root = Path(output)\n    identity = _worker_identity(new, worker_id)\n    open_run(root, identity)\n    reused, missing, files = 0, 0, {}\n    with writer_lock(root), sqlite3.connect(root / 'score_cache.sqlite3') as db:\n        db.execute('CREATE TABLE IF NOT EXISTS scores(key TEXT PRIMARY KEY,payload TEXT NOT NULL)')\n        for entry in new['packets']:\n            if entry['worker_id'] != worker_id:\n                continue\n            packet = load_packet(new_dir, entry)\n            previous = old_entries.get(entry['id'])\n            require(previous is not None, 'Expanded run has an unknown query')\n            original = load_packet(old_dir, previous)\n            require(original['query'] == packet['query'], 'Query text changed')\n            old_worker = Path(old_workers) / f\"worker_{previous['worker_id']}\"\n            manifest = json.loads((old_worker / 'manifest.json').read_text(encoding='utf-8'))\n            expected = _worker_identity(old, previous['worker_id'])\n            require(manifest['identity'] == expected and manifest['signature'] == text_hash(canonical_json(expected))\n                    and manifest['state'] == 'complete', 'Old worker not complete or mismatched')\n            path = old_worker / f\"query-{previous['id']}.json\"\n            rows = verify_scores(checked(path), original, previous, old, manifest['signature'])\n            files[str(previous['id'])] = file_hash(path)\n            scores = {}\n            for item, value in zip(original['candidates'], rows):\n                key = score_key(original['query'], item['retrieval_text'], policy)\n                payload = {k: v for k, v in value.items() if k != 'chunk_id'}\n                require(key not in scores or scores[key] == payload, 'Conflicting old scores for identical text')\n                scores[key] = payload\n            for item in packet['candidates']:\n                key = score_key(packet['query'], item['retrieval_text'], policy)\n                if key not in scores:\n                    missing += 1\n                    continue\n                payload = canonical_json(scores[key])\n                hit = db.execute('SELECT payload FROM scores WHERE key=?', (key,)).fetchone()\n                require(hit is None or json.loads(hit[0]) == scores[key], 'Existing cache conflicts with verified score')\n                db.execute('INSERT OR IGNORE INTO scores VALUES(?,?)', (key, payload))\n                reused += 1\n            db.commit()\n    report = dict(old_signature=old['signature'], new_signature=new['signature'],\n                  reused_pairs=reused, new_pairs=missing, old_score_files=files,\n                  helper_sha256=file_hash(__file__))\n    write_json(root / 'REUSE.json', report)\n    return report\n\n\ndef package(results, query_ids, allowed_ids, destination):\n    \"\"\"Package only source-validated results with exact query and ID coverage.\"\"\"\n    results, destination = Path(results), Path(destination)\n    report = json.loads((results / 'VALIDATION.json').read_text(encoding='utf-8'))\n    require(report['validation'] == 'passed' and report['original_evidence_verified'] is True,\n            'Original source validation is required')\n    require(file_hash(results / 'manifest.json') == report['manifest_sha256'], 'Result manifest changed')\n    path = results / 'predictions.json'\n    require(file_hash(path) == report['predictions_sha256'], 'Prediction hash differs')\n    payload = path.read_bytes()\n    rows = json.loads(payload)\n    require([r['id'] for r in rows] == query_ids and len(set(query_ids)) == len(query_ids), 'Query coverage differs')\n    for row in rows:\n        docs = row['relevant_docs']\n        require(all(type(d) is int and d in allowed_ids for d in docs) and len(docs) == len(set(docs)), 'Invalid document IDs')\n        for chunk in row['relevant_chunks']:\n            require(type(chunk['doc_id']) is int and chunk['doc_id'] in docs and\n                    isinstance(chunk['chunk_text'], str) and chunk['chunk_text'].strip(), 'Invalid evidence')\n    destination.mkdir(parents=True, exist_ok=False)\n    (destination / 'submission.json').write_bytes(payload)\n    archive = destination / 'submission.zip'\n    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:\n        z.writestr('submission.json', payload)\n    with zipfile.ZipFile(archive) as z:\n        require(z.namelist() == ['submission.json'] and z.read('submission.json') == payload, 'ZIP differs')\n    write_json(destination / 'REPORT.json', dict(queries=len(rows), source_validation=report,\n               submission_sha256=file_hash(path), zip_sha256=file_hash(archive), relevance_measured=False))\n    return archive\n\r\n"
(PROJECT / 'stage9_expand_support.py').write_text(HELPER_SOURCE, encoding='utf-8')
from stage9_expand_support import seed_cache, package
from vibiomir.stage9_pipeline import prepare, bundle, export, validate_results, validate_worker
from vibiomir.io import write_json
def restore_bundle(archive_path, target):
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as z:
        meta = json.loads(z.read('BUNDLE.json'))
        assert digest(canonical({k:v for k,v in meta.items() if k != 'signature'}).encode()) == meta['signature']
        assert len(z.namelist()) == len(set(z.namelist())) and set(z.namelist()) == set(meta['files']) | {'BUNDLE.json'}
        for name, expected in meta['files'].items():
            path = (target / name).resolve()
            assert target.resolve() in path.parents
            data = z.read(name)
            assert digest(data) == expected
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(data)
import time
import threading
from contextlib import contextmanager

@contextmanager
def progress(label):
    started = time.monotonic()
    finished = threading.Event()
    def heartbeat():
        while not finished.wait(30):
            print(f'{label}: still running, {(time.monotonic()-started)/60:.1f} min', flush=True)
    thread = threading.Thread(target=heartbeat, daemon=True)
    print(label, flush=True)
    thread.start()
    try:
        yield
    finally:
        finished.set()
        thread.join()
        print(f'{label}: stopped after {(time.monotonic()-started)/60:.1f} min', flush=True)

def stage_store(name):
    destination = LOCAL_INPUTS / name
    is_chunks = name == 'chunks_stage5_full_structure'
    def check_snapshot(folder):
        if is_chunks:
            from vibiomir.retrieval_contracts import chunk_snapshot
            actual = chunk_snapshot(folder)
            expected = state['identity']['source']
            assert actual == expected, (f'Chunk store does not match frozen S09: {folder}\n'
                f'Actual: {canonical(actual)}\nExpected: {canonical(expected)}\n'
                'Use the original structural snapshot; equal batch counts alone are insufficient.')
    override = globals().get('CHUNK_STORE_OVERRIDE') if is_chunks else None
    if override is not None:
        override = Path(override)
        check_snapshot(override)  # Check before copying a large store.
    if (destination / '.copy_complete').is_file():
        check_snapshot(destination)
        return destination  # prepare/validation rechecks payloads
    extracted = override if override is not None else next((p for p in (ARTIFACTS_DIR / name, ROOT / name) if (p / 'manifest.json').is_file()), None)
    if extracted:
        check_snapshot(extracted)
        if extracted.resolve() == destination.resolve():
            return destination
        with progress(f'Copy {name}: {extracted} -> {destination}'):
            entries = [p for p in extracted.rglob('*') if p.is_file()]
            total = sum(p.stat().st_size for p in entries)
            assert shutil.disk_usage('/content').free > total + 8*1024**3, 'Insufficient local disk'
            copied = 0
            for number, src in enumerate(entries, 1):
                dst = destination / src.relative_to(extracted)
                dst.parent.mkdir(parents=True, exist_ok=True)
                # Copy again after interruption; existence alone does not prove completeness.
                shutil.copy2(src, dst)
                copied += src.stat().st_size
                if number % 50 == 0 or number == len(entries):
                    print(f'{name}: {number}/{len(entries)} files, '
                          f'{copied/1024**3:.2f}/{total/1024**3:.2f} GiB', flush=True)
    else:
        assert ARTIFACTS_ZIP.is_file(), f'Missing store {name}; provide extracted store or current artifacts.zip.'
        with zipfile.ZipFile(ARTIFACTS_ZIP) as z:
            prefixes = [n[:-len('manifest.json')] for n in z.namelist() if n.endswith(name + '/manifest.json')]
            assert len(prefixes) == 1, f'Expected one {name} store in artifacts.zip, found {len(prefixes)}'
            members = [i for i in z.infolist() if i.filename.startswith(prefixes[0]) and not i.is_dir()]
            assert shutil.disk_usage('/content').free > sum(i.file_size for i in members) + 8*1024**3
            for info in members:
                path = (destination / info.filename[len(prefixes[0]):]).resolve()
                assert destination.resolve() in path.parents
                path.parent.mkdir(parents=True, exist_ok=True)
                with z.open(info) as src, path.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
    assert (destination / 'VALIDATION.json').is_file(), f'Missing validation for {name}'
    check_snapshot(destination)
    (destination / '.copy_complete').write_text('Copied; payloads are checked by the pipeline.')
    return destination
LOCAL_INPUTS.mkdir(parents=True, exist_ok=True)
chunk_dir = stage_store('chunks_stage5_full_structure')
# Keep local absolute paths stable across Colab restarts.
fixed_query = LOCAL_INPUTS / 'query.parquet'
shutil.copy2(query_path, fixed_query)
checkpoint = RUN_DRIVE / 'expanded_prepared.zip'
if checkpoint.is_file():
    restore_bundle(checkpoint, NEW_PREPARED)
    expanded = prepared(NEW_PREPARED)
else:
    config = json.loads(json.dumps(state['identity']['settings']))
    config.update(queries=str(fixed_query), chunk_dir=str(chunk_dir),
                  output_dir=str(NEW_PREPARED), rerank_top_k=RERANK_TOP_K,
                  workers=1, selection=LIMITS)
    for branch in config['branches']:
        name = Path(branch['candidate_dir']).name
        branch['candidate_dir'] = str(stage_store(name))
    config_path = LOCAL_INPUTS / 'expanded_config.json'
    write_json(config_path, config)
    with progress('Prepare expanded candidates on CPU'):
        expanded = prepare(config_path)
    local_zip = LOCAL_INPUTS / 'expanded_prepared.zip'
    bundle(NEW_PREPARED, local_zip)
    pending = checkpoint.with_suffix('.zip.tmp')
    shutil.copy2(local_zip, pending)
    assert file_hash(pending) == file_hash(local_zip)
    pending.replace(checkpoint)
assert expanded['identity']['settings']['rerank_top_k'] == RERANK_TOP_K
assert expanded['identity']['settings']['selection'] == LIMITS
assert expanded['identity']['settings']['workers'] == 1
assert expanded['identity']['query_file_sha256'] == file_hash(fixed_query)
assert len(expanded['packets']) == EXPECTED_QUERIES
# Normalize the already verified old worker layout with local symlinks.
old_layout = PROJECT / 'old_workers'
old_layout.mkdir(exist_ok=True)
for i, folder in worker_roots.items():
    link = old_layout / f'worker_{i}'
    if not link.exists(): link.symlink_to(folder.resolve(), target_is_directory=True)
with progress('Verify and reuse old scores'):
    reuse = seed_cache(PROJECT, old_layout, NEW_PREPARED, NEW_WORKERS / 'worker_0')
print({k: reuse[k] for k in ('reused_pairs', 'new_pairs')})
write_json(RUN_DRIVE / 'REUSE.json', reuse)
old_by_id = {x['id']:x for x in state['packets']}
diversity = []
for entry in expanded['packets']:
    new_packet = load_packet(NEW_PREPARED, entry)
    old_packet = load_packet(PROJECT, old_by_id[entry['id']])
    assert [x['chunk_id'] for x in new_packet['candidates'][:len(old_packet['candidates'])]] == [x['chunk_id'] for x in old_packet['candidates']], 'Pool prefix changed; this is not a pool-size-only experiment.'
    diversity.append(dict(id=entry['id'], old_documents=len({x['doc_id'] for x in old_packet['candidates']}),
                          new_documents=len({x['doc_id'] for x in new_packet['candidates']})))
write_json(RUN_DRIVE / 'POOL_DIVERSITY.json', diversity)
print('Prepared', expanded['candidates'], 'pairs. Check REUSE.json and POOL_DIVERSITY.json.')


In [ ]:
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements-stage9-colab.txt')], check=True)
import torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime.'
from vibiomir.stage9_reranker import recommended_runtime
preset = recommended_runtime(torch.cuda.get_device_properties(0).total_memory / 1024**3)
print(torch.cuda.get_device_name(0), preset)


In [ ]:
# Rerun this cell with STOP_AFTER_QUERIES=None after checking pilot time.
import time
command = [sys.executable, '-u', '-m', 'vibiomir.stage9_pipeline', 'rerank-worker',
           '--prepared-dir', str(NEW_PREPARED), '--output-dir', str(NEW_WORKERS / 'worker_0'),
           '--worker-id', '0', '--backup-dir', str(RUN_DRIVE / 'workers/worker_0'),
           '--batch-size', str(preset['batch_size']), '--padded-tokens', str(preset['padded_tokens'])]
if STOP_AFTER_QUERIES is not None:
    command += ['--stop-after-queries', str(STOP_AFTER_QUERIES)]
started = time.monotonic()
log = RUN_DRIVE / 'latest_worker.log'
with log.open('w', encoding='utf-8') as stream:
    process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            stream.write(line); stream.flush()
        status = process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
assert status == 0, f'Worker failed; inspect {log}. Completed query backups are retained.'
result = json.loads((NEW_WORKERS / 'worker_0/manifest.json').read_text())
print('Elapsed seconds:', round(time.monotonic()-started, 1), 'State:', result['state'], 'Queries:', result['queries'])
if result['state'] != 'complete':
    print('Pilot/partial run: set STOP_AFTER_QUERIES=None and rerun this GPU cell. Do not export yet.')


In [ ]:
# Full source validation and official-ID checks before packaging.
validate_worker(NEW_PREPARED, NEW_WORKERS / 'worker_0', 0)
RESULTS = Path('/content/stage9_expand_results')
export(NEW_PREPARED, RESULTS, mode='reranked', workers_dir=NEW_WORKERS)
validation = validate_results(NEW_PREPARED, RESULTS, chunk_dir=chunk_dir,
                              queries_path=fixed_query, workers_dir=NEW_WORKERS)
from datetime import datetime, timezone
destination = RUN_DRIVE / ('submission_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
archive = package(RESULTS, [q['id'] for q in official_queries], allowed, destination)
shutil.copy2(RESULTS / 'VALIDATION.json', destination / 'VALIDATION.json')
shutil.copy2(RESULTS / 'manifest.json', destination / 'result_manifest.json')
# Compare output coverage with the same 120/120/6 policy on the old pool.
new_predictions = {r['id']:r for r in json.loads((RESULTS / 'predictions.json').read_text())}
comparison = []
for entry in state['packets']:
    packet = load_packet(PROJECT, entry)
    worker_id = entry['worker_id']
    score_file = worker_roots[worker_id] / f"query-{entry['id']}.json"
    scores = [r['score'] for r in verify_scores(checked(score_file), packet, entry, state, worker_manifests[worker_id]['signature'])]
    baseline, _ = select(packet['candidates'], scores, LIMITS)
    current = new_predictions[entry['id']]
    before, after = set(baseline['relevant_docs']), set(current['relevant_docs'])
    comparison.append(dict(id=entry['id'], old_documents=len(before), new_documents=len(after),
                           added_documents=len(after-before), removed_documents=len(before-after),
                           old_chunks=len(baseline['relevant_chunks']), new_chunks=len(current['relevant_chunks'])))
write_json(destination / 'COMPARISON.json', dict(reference_policy=LIMITS, queries=comparison,
           relevance_measured=False, note='Coverage changes are not relevance gains.'))
print('Queries with changed document sets:', sum(bool(r['added_documents'] or r['removed_documents']) for r in comparison))
print('Validated submission:', archive)
print('Relevance is not measured here. Compare against 0.1626 only in the same scoring phase.')
from google.colab import files
files.download(str(archive))
